# PoC SD vs DES (Supply Chain)

Ce notebook reproduit le PoC: 
- equivalence SD/DES sur un cas agrege deterministe,
- apport DES pour le risque operationnel (variabilite, disruptions),
- apport SD pour l'exploration rapide de politiques.

## 1) Imports et configuration

In [ ]:
%matplotlib inline
from dataclasses import replace
import random
import statistics
import time

import numpy as np
import matplotlib.pyplot as plt

from main_system_dynamics import SystemDynamicsConfig, run_simulation, slugify
from main_supply_chain_des import DESConfig, SupplyChainDES
from compare_sd_des import StochasticSupplyChainDES
from supply_graph_tools import derive_supply_graph_summary

HORIZON = 180
STEP_DAY = 60
STEP_DEMAND = 80.0
INITIAL_DEMAND = 50.0

summary = derive_supply_graph_summary('knowledge_graph.json', 'actor_coords.json')
t1 = summary.suggested_transport_days['transport_1_days']
t2 = summary.suggested_transport_days['transport_2_days']
t3 = summary.suggested_transport_days['transport_3_days']

print('Transport days derives du graphe:', summary.suggested_transport_days)
print('Flows detectes:', summary.flow_counts)

sd_cfg = SystemDynamicsConfig(
    horizon_days=HORIZON,
    demand_step_day=STEP_DAY,
    demand_step_value=STEP_DEMAND,
    transport_1_days=t1,
    transport_2_days=t2,
    transport_3_days=t3,
)

des_cfg = DESConfig(
    horizon_days=HORIZON,
    demand_step_day=STEP_DAY,
    demand_step_value=STEP_DEMAND,
    enable_dynamics=True,
    transport_1_days=t1,
    transport_2_days=t2,
    transport_3_days=t3,
)

stock_key = f"inventory_{slugify('stock_arrivee_ligne_production')}"


## 2) Equivalence SD vs DES (deterministe)

In [ ]:
sd_series = run_simulation(sd_cfg).series
des_series = SupplyChainDES(des_cfg).run().series

stage_names = [
    'extraction',
    'stock_amont',
    'stock_apres_transport_amont',
    'transformation_1',
    'stock_apres_transformation_1',
    'stock_apres_transport_t1',
    'transformation_2',
    'stock_distribution',
    'distribution',
    'stock_arrivee_ligne_production',
]
keys = [
    'customer_backlog',
    'customer_shipment',
    'orders_total',
    'shipments_total',
    'in_transit_total',
] + [f"inventory_{slugify(name)}" for name in stage_names]

max_diff = 0.0
for key in keys:
    d = max(abs(a - b) for a, b in zip(sd_series[key], des_series[key]))
    max_diff = max(max_diff, d)

print(f"Max |SD - DES| (deterministe): {max_diff:.3e}")

In [ ]:
days = sd_series['day']

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(days, sd_series[stock_key], label='SD stock arrivee', linewidth=2)
axes[0].step(days, des_series[stock_key], where='post', label='DES stock arrivee', linewidth=1.6)
axes[0].set_ylabel('Unites')
axes[0].set_title('Cas deterministe: equivalence SD / DES')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(days, sd_series['customer_backlog'], label='SD backlog')
axes[1].step(days, des_series['customer_backlog'], where='post', label='DES backlog')
axes[1].set_xlabel('Jour')
axes[1].set_ylabel('Backlog')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 3) Apport DES: risque operationnel (stochastique)

In [ ]:
REPS = 30
SEED = 7

delay_noise = 0.30
disruption_prob = 0.08
disruption_factor = 0.50

rng_master = random.Random(SEED)
backlog_runs = []
service_levels = []
max_backlogs = []

for _ in range(REPS):
    seed = rng_master.randint(1, 10_000_000)
    rng = random.Random(seed)
    res = StochasticSupplyChainDES(
        des_cfg,
        rng=rng,
        delay_noise=delay_noise,
        disruption_prob=disruption_prob,
        disruption_factor=disruption_factor,
    ).run()
    s = res.series
    backlog_runs.append(s['customer_backlog'])
    total_demand = sum(s['customer_demand'])
    total_ship = sum(s['customer_shipment'])
    service_levels.append((total_ship / total_demand) if total_demand > 0 else 1.0)
    max_backlogs.append(max(s['customer_backlog']))

backlog_arr = np.array(backlog_runs)
p10 = np.percentile(backlog_arr, 10, axis=0)
p50 = np.percentile(backlog_arr, 50, axis=0)
p90 = np.percentile(backlog_arr, 90, axis=0)

print(f"DES stochastique - P95 max backlog: {np.percentile(max_backlogs, 95):.1f} u")
print(f"DES stochastique - Service median: {100*np.percentile(service_levels, 50):.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(days, p10, p90, alpha=0.25, label='DES stochastique P10-P90')
ax.plot(days, p50, label='DES stochastique mediane')
ax.plot(days, sd_series['customer_backlog'], '--', label='SD backlog deterministe')
ax.set_title('Apport DES: distribution du risque backlog')
ax.set_xlabel('Jour')
ax.set_ylabel('Backlog client')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4) Apport SD: exploration rapide des politiques

In [ ]:
adjust_times = [6, 8, 10, 12, 15, 18, 22, 28]

t0 = time.perf_counter()
sd_max_backlog = []
sd_avg_backlog = []
for tau in adjust_times:
    cfg_tau = replace(sd_cfg, inventory_adjust_time=float(tau))
    s = run_simulation(cfg_tau).series
    sd_max_backlog.append(max(s['customer_backlog']))
    sd_avg_backlog.append(statistics.fmean(s['customer_backlog']))
sd_elapsed = time.perf_counter() - t0

print(f"SD sweep: {len(adjust_times)} policies en {sd_elapsed:.3f}s")

In [ ]:
des_points = [8, 15, 22]
des_reps = 6
rng_master = random.Random(SEED + 1234)

t0 = time.perf_counter()
des_mean_max_backlog = []
for tau in des_points:
    vals = []
    for _ in range(des_reps):
        seed = rng_master.randint(1, 10_000_000)
        rng = random.Random(seed)
        cfg_tau = replace(des_cfg, inventory_adjust_time=float(tau))
        res = StochasticSupplyChainDES(
            cfg_tau,
            rng=rng,
            delay_noise=delay_noise,
            disruption_prob=disruption_prob,
            disruption_factor=disruption_factor,
        ).run()
        vals.append(max(res.series['customer_backlog']))
    des_mean_max_backlog.append(statistics.fmean(vals))
des_elapsed = time.perf_counter() - t0

print(f"DES sweep indicatif: {len(des_points)} policies x {des_reps} reps en {des_elapsed:.3f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(adjust_times, sd_max_backlog, marker='o', label='SD max backlog')
ax.plot(adjust_times, sd_avg_backlog, marker='o', label='SD backlog moyen')
ax.scatter(des_points, des_mean_max_backlog, marker='x', s=90, label=f'DES mean max backlog ({des_reps} reps/point)')
ax.set_title('Apport SD: balayage plus rapide des politiques')
ax.set_xlabel('Inventory adjust time (jours)')
ax.set_ylabel('Backlog (unites)')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5) Analyse de perturbation (SD compare DES)

On applique une perturbation impulsionnelle de demande (pulse) et on compare l'ecart a une trajectoire de reference.

In [ ]:
PERT_DAY = 90
PERT_DURATION = 7
PERT_AMP = 25.0

sd_cfg_pert = replace(sd_cfg, demand_step_day=HORIZON + 1, demand_step_value=INITIAL_DEMAND)
des_cfg_pert = replace(des_cfg, demand_step_day=HORIZON + 1, demand_step_value=INITIAL_DEMAND, enable_dynamics=True)

def baseline_demand(day, cfg):
    return INITIAL_DEMAND

def pulse_demand(day, cfg):
    d = INITIAL_DEMAND
    if PERT_DAY <= day < PERT_DAY + PERT_DURATION:
        d += PERT_AMP
    return d

sd_base = run_simulation(sd_cfg_pert, demand_fn=baseline_demand).series
sd_pulse = run_simulation(sd_cfg_pert, demand_fn=pulse_demand).series

des_base = SupplyChainDES(des_cfg_pert, demand_fn=baseline_demand).run().series
des_pulse = SupplyChainDES(des_cfg_pert, demand_fn=pulse_demand).run().series

sd_delta_stock = np.array(sd_pulse[stock_key]) - np.array(sd_base[stock_key])
des_delta_stock = np.array(des_pulse[stock_key]) - np.array(des_base[stock_key])
sd_delta_backlog = np.array(sd_pulse['customer_backlog']) - np.array(sd_base['customer_backlog'])
des_delta_backlog = np.array(des_pulse['customer_backlog']) - np.array(des_base['customer_backlog'])

print(f'Peak delta backlog SD: {np.max(sd_delta_backlog):.1f} u')
print(f'Peak delta backlog DES: {np.max(des_delta_backlog):.1f} u')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(days, sd_delta_stock, label='SD delta stock arrivee', linewidth=2)
axes[0].step(days, des_delta_stock, where='post', label='DES delta stock arrivee', linewidth=1.6)
axes[0].axvspan(PERT_DAY, PERT_DAY + PERT_DURATION, alpha=0.15, color='tab:red', label='Perturbation')
axes[0].set_ylabel('Delta stock (u)')
axes[0].set_title('Reponse a perturbation - Stock arrivee ligne')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(days, sd_delta_backlog, label='SD delta backlog', linewidth=2)
axes[1].step(days, des_delta_backlog, where='post', label='DES delta backlog', linewidth=1.6)
axes[1].axvspan(PERT_DAY, PERT_DAY + PERT_DURATION, alpha=0.15, color='tab:red', label='Perturbation')
axes[1].set_ylabel('Delta backlog (u)')
axes[1].set_xlabel('Jour')
axes[1].set_title('Reponse a perturbation - Backlog client')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6) Analyse frequentielle (SD uniquement)

On fait un Bode empirique sur le modele SD (deterministe, agrege).
Pour le DES, on garde l'analyse temporelle/stochastique (sections 3 et 5).

In [ ]:
from main_system_dynamics import analyze_frequency_response

sd_freq_cfg = replace(sd_cfg, demand_step_day=HORIZON + 1, demand_step_value=INITIAL_DEMAND)

sd_fr = analyze_frequency_response(
    cfg=sd_freq_cfg,
    freq_min=0.01,
    freq_max=0.20,
    freq_points=10,
    demand_amplitude=5.0,
    warmup_cycles=3,
    measure_cycles=6,
)

print('Frequences testees (cycles/jour):', np.round(sd_fr.frequencies_cpd, 4))


In [ ]:
eps = 1e-12
sd_gain_stock_db = 20 * np.log10(np.maximum(np.array(sd_fr.gain_stock), eps))
sd_gain_backlog_db = 20 * np.log10(np.maximum(np.array(sd_fr.gain_backlog), eps))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].semilogx(sd_fr.frequencies_cpd, sd_gain_stock_db, marker='o', label='Gain stock arrivee')
axes[0].semilogx(sd_fr.frequencies_cpd, sd_gain_backlog_db, marker='o', linestyle='--', label='Gain backlog client')
axes[0].set_ylabel('Gain (dB)')
axes[0].set_title('Bode empirique - modele SD')
axes[0].grid(alpha=0.3, which='both')
axes[0].legend()

axes[1].semilogx(sd_fr.frequencies_cpd, sd_fr.phase_stock_deg, marker='o', label='Phase stock arrivee')
axes[1].semilogx(sd_fr.frequencies_cpd, sd_fr.phase_backlog_deg, marker='o', linestyle='--', label='Phase backlog client')
axes[1].set_xlabel('Frequence (cycles/jour)')
axes[1].set_ylabel('Phase (deg)')
axes[1].grid(alpha=0.3, which='both')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7) Diagnostics avances (supply chain complexe)

Objectif: pousser l'analyse frequentielle sur SD et l'analyse de distributions sur DES pour identifier les zones de risque (resonance, variabilite, queues de backlog, goulots).


In [ ]:
from advanced_supply_diagnostics import (
    AdvancedConfig,
    run_sd_frequency_analysis,
    run_des_distribution_analysis,
    build_issue_list,
    plot_sd_frequency,
    plot_des_distributions,
)

adv_cfg = AdvancedConfig(
    horizon_days=240,
    step_day=STEP_DAY,
    step_demand=STEP_DEMAND,
    transport_1_days=t1,
    transport_2_days=t2,
    transport_3_days=t3,
    replications=120,
    seed=7,
    delay_noise=0.45,
    disruption_prob=0.12,
    disruption_factor=0.45,
    freq_min=0.003,
    freq_max=0.33,
    freq_points=24,
    freq_amplitude=8.0,
    freq_warmup_cycles=4,
    freq_measure_cycles=10,
)

sd_adv = run_sd_frequency_analysis(adv_cfg)
des_adv = run_des_distribution_analysis(adv_cfg)
issues_adv = build_issue_list(sd_adv, des_adv)

plot_sd_frequency(sd_adv, 'notebook_advanced_sd_frequency.png')
plot_des_distributions(des_adv, 'notebook_advanced_des_distributions.png')

sd_sum = sd_adv['summary']
des_sum = des_adv['summary']

print('SD orders resonance gain (dB):', round(sd_sum['orders_total']['peak_gain_db'], 2))
print('SD orders resonance freq (cpd):', round(sd_sum['orders_total']['resonance_freq_cpd'], 4))
print('SD orders max distortion ratio:', round(sd_sum['orders_total']['max_distortion_ratio'], 3))
print('DES P95 max customer backlog:', round(des_sum['max_customer_backlog']['p95'], 2))
print('DES CVaR95 max customer backlog:', round(des_sum['max_customer_backlog']['cvar95'], 2))
print('DES P95 bullwhip ratio:', round(des_sum['bullwhip_ratio']['p95'], 3))
print('DES lead time mean / P95 (days):', round(des_sum['lead_time_days']['mean'], 3), '/', round(des_sum['lead_time_days']['p95'], 3))

print('\nTop bottlenecks (P95 max backlog):')
for b in des_adv['top_bottlenecks'][:5]:
    print('-', b['link_backlog_key'], '=>', round(b['p95_max_backlog'], 1))

print('\nIssues detectees:')
for item in issues_adv:
    print(f"- [{item['severity']}] {item['area']}: {item['issue']}")


## 8) Graphe de connaissance annote (localisation des problemes)

Cette vue relie les hotspots detectes (DES/SD) aux acteurs reels et aux produits, pour localiser ou les risques se concentrent dans la supply.


In [ ]:
from supply_issue_graph import plot_annotated_supply_graph
from IPython.display import Image, display

issue_map = plot_annotated_supply_graph(
    graph_path='knowledge_graph.json',
    coords_path='actor_coords.json',
    report_path='advanced_complex_full_report.json',
    output_path='notebook_supply_issue_annotated.png',
    output_json='notebook_supply_issue_annotated.json',
    output_text='notebook_supply_issue_annotated_report.txt',
)

print('Zone scores:', issue_map['zone_scores'])
print('Top links localises:')
for row in issue_map['top_localized_links'][:10]:
    print('-', row)

display(Image(filename='notebook_supply_issue_annotated.png'))


## 9) Conclusion PoC

- SD et DES sont equivalents sur le cas agrege deterministe.
- SD et DES donnent une reponse proche aux perturbations dans ce cadre deterministe.
- Le Bode est utilise sur SD (outil structurel/frequentiel).
- Pour DES, on privilegie la lecture temporelle et stochastique (risques, percentiles, reprises).
